# 34 - PF-031 a PF-038: E2E, demo final, validación y cierre documental

Este notebook cierra el backlog de producto final desde el punto de vista de evidencia y preparación de entrega.

Cubre:

- **PF-031**: suite de smoke tests end-to-end.
- **PF-032**: demo script final.
- **PF-033**: validación profesional con pantallas.
- **PF-034**: actualización documental final de capítulos 15-19.
- **PF-035**: base para capítulo 20 de conclusiones.
- **PF-036**: matriz de evidencia de defensa.
- **PF-037**: limpieza de repo y estructura final.
- **PF-038**: README general de ejecución.

No reemplaza las pruebas manuales de la demo. Deja la evidencia organizada y los comandos para ejecutarla.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json, os, textwrap, pandas as pd
from datetime import datetime

PFI_ROOT = Path('/content/drive/MyDrive/PFI_MVP')
REPO_ROOT = PFI_ROOT / 'repo'
RESULTS_ROOT = PFI_ROOT / 'results' / 'PF031_PF038_e2e_demo_validation_finalization'
DOCS_ROOT = REPO_ROOT / 'docs'
BACKLOG_ROOT = REPO_ROOT / 'backlogProducto'

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
DOCS_ROOT.mkdir(parents=True, exist_ok=True)
BACKLOG_ROOT.mkdir(parents=True, exist_ok=True)

print('PFI_ROOT:', PFI_ROOT)
print('REPO_ROOT:', REPO_ROOT, '| exists:', REPO_ROOT.exists())
print('RESULTS_ROOT:', RESULTS_ROOT)

## 1. Verificación de artefactos principales

In [ ]:
required_paths = {
    # AI service
    'python_api': REPO_ROOT / 'ai_service' / 'pfi_ai_service' / 'api.py',
    'python_agent_policy': REPO_ROOT / 'ai_service' / 'pfi_ai_service' / 'agent_policy.py',
    'python_pipeline': REPO_ROOT / 'ai_service' / 'pfi_ai_service' / 'pipeline.py',
    'python_dockerfile': REPO_ROOT / 'ai_service' / 'Dockerfile',

    # Config
    'data_freeze_config': REPO_ROOT / 'config' / 'data_freeze_config.json',
    'model_registry_final': REPO_ROOT / 'config' / 'model_registry_final.json',

    # Backend
    'backend_root': REPO_ROOT / 'backend_product_spring',
    'backend_pom': REPO_ROOT / 'backend_product_spring' / 'pom.xml',
    'backend_controller': REPO_ROOT / 'backend_product_spring' / 'src' / 'main' / 'java' / 'ar' / 'edu' / 'uade' / 'pfi' / 'backend' / 'controller' / 'AiBackendController.java',

    # Frontend
    'frontend_root': REPO_ROOT / 'frontend_product_ui',
    'frontend_app': REPO_ROOT / 'frontend_product_ui' / 'src' / 'App.tsx',
    'frontend_api': REPO_ROOT / 'frontend_product_ui' / 'src' / 'api.ts',
    'frontend_package': REPO_ROOT / 'frontend_product_ui' / 'package.json',

    # Docs by block
    'pf012_docs': DOCS_ROOT / 'PF012_PF019_service_endpoints_agent.md',
    'pf020_backend_docs': DOCS_ROOT / 'PF020_PF025_spring_backend_contract.md',
    'pf020_frontend_docs': DOCS_ROOT / 'PF020_PF025_frontend_backend_contract.md',
    'pf026_frontend_docs': DOCS_ROOT / 'PF026_PF030_frontend_product_ui.md',
    'pf026_preview': DOCS_ROOT / 'PF026_PF030_frontend_static_preview.html',
}

artifact_rows = []
for key, path in required_paths.items():
    artifact_rows.append({
        'artifact_key': key,
        'path': str(path),
        'exists': path.exists(),
        'size_bytes': path.stat().st_size if path.exists() and path.is_file() else None
    })

artifact_df = pd.DataFrame(artifact_rows)
artifact_path = RESULTS_ROOT / 'PF031_artifact_readiness_inventory.csv'
artifact_df.to_csv(artifact_path, index=False)
print('Artifact readiness inventory:', artifact_path)
artifact_df

## 2. Checklist E2E y comandos de ejecución

In [ ]:
e2e_steps = [
    {
        'order': 1,
        'layer': 'Python AI service',
        'command': 'cd ai_service && uvicorn pfi_ai_service.api:app --host 0.0.0.0 --port 8000',
        'expected': 'GET http://localhost:8000/health devuelve status ok',
        'evidence': 'captura terminal + endpoint /health'
    },
    {
        'order': 2,
        'layer': 'Spring Boot backend',
        'command': 'cd backend_product_spring && ./mvnw spring-boot:run  # o mvn spring-boot:run',
        'expected': 'GET http://localhost:8080/api/ai/health responde usando el servicio Python',
        'evidence': 'captura terminal + endpoint /api/ai/health'
    },
    {
        'order': 3,
        'layer': 'Frontend React',
        'command': 'cd frontend_product_ui && npm install && npm run dev',
        'expected': 'Vite levanta la UI y consume VITE_API_BASE_URL=http://localhost:8080',
        'evidence': 'captura navegador dashboard'
    },
    {
        'order': 4,
        'layer': 'Pipeline run',
        'command': 'Desde UI: ejecutar caso demo / desde backend: POST /api/ai/pipeline/run',
        'expected': 'Se genera runId, decisión del agente, mediciones y overlay/evidencia',
        'evidence': 'captura detalle de caso'
    },
    {
        'order': 5,
        'layer': 'Professional review',
        'command': 'Cambiar estado de revisión y guardar observación',
        'expected': 'PATCH /api/ai/review/{runId} actualiza estado: pendiente/aceptado/observado/descartado',
        'evidence': 'captura UI + respuesta backend'
    },
]
e2e_df = pd.DataFrame(e2e_steps)
e2e_path = RESULTS_ROOT / 'PF031_e2e_smoke_test_plan.csv'
e2e_df.to_csv(e2e_path, index=False)
print('E2E smoke test plan:', e2e_path)
e2e_df

In [ ]:
e2e_md = f'''# PF-031 - Smoke tests end-to-end

Fecha de generación: {datetime.now().isoformat(timespec="seconds")}

## Objetivo

Validar la cadena completa del MVP técnico:

```text
Python FastAPI IA → Spring Boot backend → React frontend → revisión profesional
```

## Comandos sugeridos

### 1. Servicio Python

```bash
cd ai_service
uvicorn pfi_ai_service.api:app --host 0.0.0.0 --port 8000
```

Validar:

```bash
curl http://localhost:8000/health
curl http://localhost:8000/models
```

### 2. Backend Spring Boot

```bash
cd backend_product_spring
mvn spring-boot:run
```

Validar:

```bash
curl http://localhost:8080/api/ai/health
curl http://localhost:8080/api/ai/models
```

### 3. Frontend React

```bash
cd frontend_product_ui
npm install
npm run dev
```

Configurar `.env`:

```bash
VITE_API_BASE_URL=http://localhost:8080
```

## Escenario mínimo de prueba

1. Abrir dashboard.
2. Ver estado de backend/modelos.
3. Ejecutar pipeline demo.
4. Abrir detalle del caso.
5. Revisar flags, mediciones y overlay/evidencia.
6. Guardar estado de revisión.
7. Confirmar que la salida mantiene `human_review_required=true`.

## Resultado esperado

La demo debe mostrar un sistema asistivo, no diagnóstico autónomo.
'''

e2e_md_path = DOCS_ROOT / 'PF031_e2e_smoke_tests.md'
e2e_md_path.write_text(e2e_md, encoding='utf-8')
print('Wrote:', e2e_md_path)

## 3. Guion de demo final

In [ ]:
demo_steps = [
    {'order': 1, 'section': 'Contexto', 'speaker_goal': 'Explicar el problema y alcance asistivo', 'screen': 'Portada / arquitectura', 'key_message': 'El sistema apoya revisión de RM lumbar; no diagnostica automáticamente.'},
    {'order': 2, 'section': 'Arquitectura', 'speaker_goal': 'Mostrar cadena técnica', 'screen': 'Diagrama Python → Spring → React', 'key_message': 'La IA corre en Python; Spring orquesta producto; React presenta evidencia.'},
    {'order': 3, 'section': 'Modelos', 'speaker_goal': 'Mostrar modelos finales', 'screen': 'Registry/modelos', 'key_message': 'Se consolidaron modelos axial T2 y sagital SPIDER con métricas documentadas.'},
    {'order': 4, 'section': 'Pipeline', 'speaker_goal': 'Ejecutar caso demo', 'screen': 'Frontend dashboard', 'key_message': 'El usuario profesional inicia/revisa una ejecución trazable.'},
    {'order': 5, 'section': 'Resultado IA', 'speaker_goal': 'Mostrar decisión del agente', 'screen': 'Detalle de caso', 'key_message': 'El agente prioriza revisión y expone flags/confianza.'},
    {'order': 6, 'section': 'Evidencia visual', 'speaker_goal': 'Mostrar overlay/mediciones', 'screen': 'Panel overlay', 'key_message': 'La salida es evidencia visual y mediciones mínimas, no diagnóstico.'},
    {'order': 7, 'section': 'Human-in-the-loop', 'speaker_goal': 'Guardar revisión', 'screen': 'Formulario de revisión', 'key_message': 'El profesional acepta, observa o descarta la salida.'},
    {'order': 8, 'section': 'Cierre', 'speaker_goal': 'Explicar límites y próximos pasos', 'screen': 'Roadmap', 'key_message': 'Falta validación clínica mayor, visor avanzado y posible 3D futuro.'},
]
demo_df = pd.DataFrame(demo_steps)
demo_path = RESULTS_ROOT / 'PF032_demo_script_steps.csv'
demo_df.to_csv(demo_path, index=False)

demo_md = '# PF-032 - Guion de demo final\n\n'
for row in demo_steps:
    demo_md += f"## {row['order']}. {row['section']}\n\n"
    demo_md += f"- Objetivo: {row['speaker_goal']}\n"
    demo_md += f"- Pantalla: {row['screen']}\n"
    demo_md += f"- Mensaje clave: {row['key_message']}\n\n"

demo_md += '''## Frase de defensa recomendada

El aporte del producto no es competir con una IA comercial aislada, sino integrar modelos de segmentación axial y sagital, un pipeline de inferencia, un agente de priorización, un backend de producto y una interfaz de revisión profesional dentro de un flujo trazable y human-in-the-loop.
'''

demo_md_path = DOCS_ROOT / 'PF032_final_demo_script.md'
demo_md_path.write_text(demo_md, encoding='utf-8')

print('Demo script CSV:', demo_path)
print('Demo script MD:', demo_md_path)
demo_df

## 4. Plantilla de validación profesional

In [ ]:
validation_questions = [
    {'dimension': 'Comprensión', 'question': '¿La pantalla permite entender qué estructura o plano se está revisando?', 'scale': '1-5', 'notes_field': 'comentario_abierto'},
    {'dimension': 'Utilidad', 'question': '¿La priorización del agente sería útil para ordenar una revisión?', 'scale': '1-5', 'notes_field': 'comentario_abierto'},
    {'dimension': 'Confianza', 'question': '¿Qué información adicional necesitaría para confiar en la segmentación?', 'scale': 'abierta', 'notes_field': 'respuesta_textual'},
    {'dimension': 'Riesgo', 'question': '¿Qué elemento de la interfaz podría inducir a error o sobrerreliance?', 'scale': 'abierta', 'notes_field': 'respuesta_textual'},
    {'dimension': 'Flujo', 'question': '¿El estado pendiente/aceptado/observado/descartado representa una revisión profesional razonable?', 'scale': '1-5', 'notes_field': 'comentario_abierto'},
    {'dimension': 'Primera versión', 'question': '¿Qué funcionalidad considera indispensable para una primera versión usable?', 'scale': 'abierta', 'notes_field': 'respuesta_textual'},
    {'dimension': 'Valor general', 'question': 'Como herramienta de apoyo, ¿qué valor le asignaría al MVP actual?', 'scale': '1-5', 'notes_field': 'comentario_abierto'},
]
validation_df = pd.DataFrame(validation_questions)
validation_path = RESULTS_ROOT / 'PF033_professional_validation_questions.csv'
validation_df.to_csv(validation_path, index=False)

validation_md = '''# PF-033 - Plantilla de validación profesional con pantallas

## Instrucción para la sesión

Se presenta un MVP técnico de apoyo a la revisión de resonancia magnética lumbar. La herramienta no realiza diagnóstico clínico autónomo. El objetivo de la sesión es evaluar claridad, utilidad, riesgos y requerimientos mínimos para una versión futura.

## Preguntas

'''

for i, row in enumerate(validation_questions, start=1):
    validation_md += f"{i}. **{row['dimension']}**: {row['question']}\n"
    validation_md += f"   - Tipo de respuesta: {row['scale']}\n\n"

validation_md += '''## Observaciones finales

- Especialidad del profesional:
- Relación con RM lumbar:
- Comentarios adicionales:
- Recomendación de mejora prioritaria:
'''

validation_md_path = DOCS_ROOT / 'PF033_professional_validation_template.md'
validation_md_path.write_text(validation_md, encoding='utf-8')

print('Validation questions CSV:', validation_path)
print('Validation template MD:', validation_md_path)
validation_df

## 5. Plan de actualización documental final

In [ ]:
doc_update_rows = [
    {'chapter': '15', 'title': 'Implementación técnica del pipeline IA', 'update_needed': 'Actualizar de checkpoint a producto final incorporando PF001-PF019 y módulos productivos.', 'priority': 'alta'},
    {'chapter': '16', 'title': 'Resultados de segmentación y validación técnica', 'update_needed': 'Mantener métricas finales PF004-PF007 y aclarar registry/modelos finales.', 'priority': 'alta'},
    {'chapter': '17', 'title': 'Inferencia multiplanar y agente', 'update_needed': 'Incorporar endpoints PF012-PF019 y política endurecida del agente.', 'priority': 'alta'},
    {'chapter': '18', 'title': 'Integración del MVP técnico', 'update_needed': 'Actualizar con backend Spring Boot PF020-PF025 y frontend PF026-PF030.', 'priority': 'alta'},
    {'chapter': '19', 'title': 'Discusión, alcance y limitaciones', 'update_needed': 'Ajustar alcance final, riesgos residuales, validación pendiente y diferencias con solución comercial.', 'priority': 'media'},
    {'chapter': '20', 'title': 'Conclusiones', 'update_needed': 'Redactar capítulo nuevo con cierre técnico, metodológico, producto y trabajo futuro.', 'priority': 'alta'},
]
doc_update_df = pd.DataFrame(doc_update_rows)
doc_update_path = RESULTS_ROOT / 'PF034_PF035_documentation_update_plan.csv'
doc_update_df.to_csv(doc_update_path, index=False)

doc_update_md = '''# PF-034 / PF-035 - Plan de actualización documental final

Este plan indica cómo pasar los capítulos checkpoint a capítulos finales de entrega.

'''
for row in doc_update_rows:
    doc_update_md += f"## Capítulo {row['chapter']} - {row['title']}\n\n"
    doc_update_md += f"- Actualización requerida: {row['update_needed']}\n"
    doc_update_md += f"- Prioridad: {row['priority']}\n\n"

doc_update_md += '''## Nota editorial

Los capítulos finales deben evitar presentar el sistema como diagnóstico autónomo. La narrativa correcta es: MVP técnico asistivo, trazable, con revisión profesional obligatoria.
'''

doc_update_md_path = DOCS_ROOT / 'PF034_PF035_documentation_update_plan.md'
doc_update_md_path.write_text(doc_update_md, encoding='utf-8')

print('Documentation update plan:', doc_update_path)
print('Documentation update MD:', doc_update_md_path)
doc_update_df

## 6. Matriz de evidencia de defensa

In [ ]:
evidence_rows = [
    {'evidence_area': 'Datos', 'artifact': 'PF001-PF003 dataset freeze', 'proof': 'data_freeze_config, inventario, splits', 'defense_value': 'Reproducibilidad y alcance claro.'},
    {'evidence_area': 'Modelos', 'artifact': 'PF004-PF007 final models', 'proof': 'model_registry_final, checkpoints, métricas', 'defense_value': 'Modelos finales trazables.'},
    {'evidence_area': 'Inferencia', 'artifact': 'PF008-PF011 modules', 'proof': 'preprocessing, inference, overlay, measurements, pipeline', 'defense_value': 'Lógica fuera de notebooks.'},
    {'evidence_area': 'Servicio IA', 'artifact': 'PF012-PF019 FastAPI endpoints', 'proof': 'endpoint smoke tests', 'defense_value': 'Servicio IA consumible.'},
    {'evidence_area': 'Backend', 'artifact': 'PF020-PF025 Spring Boot', 'proof': 'controller, DTOs, WebClient, review store', 'defense_value': 'Backend de producto.'},
    {'evidence_area': 'Frontend', 'artifact': 'PF026-PF030 React UI', 'proof': 'dashboard, worklist, detalle, overlay, revisión', 'defense_value': 'Producto demostrable.'},
    {'evidence_area': 'Validación', 'artifact': 'PF033 template', 'proof': 'preguntas y escala profesional', 'defense_value': 'Camino claro de validación con usuarios.'},
    {'evidence_area': 'Metodología', 'artifact': 'backlogProducto cierres', 'proof': 'planes y cierres PF', 'defense_value': 'Trazabilidad incremental.'},
]
evidence_df = pd.DataFrame(evidence_rows)
evidence_path = RESULTS_ROOT / 'PF036_defense_evidence_matrix.csv'
evidence_df.to_csv(evidence_path, index=False)

evidence_md = '# PF-036 - Matriz de evidencia de defensa\n\n'
for row in evidence_rows:
    evidence_md += f"## {row['evidence_area']}\n\n"
    evidence_md += f"- Artefacto: {row['artifact']}\n"
    evidence_md += f"- Prueba: {row['proof']}\n"
    evidence_md += f"- Valor para defensa: {row['defense_value']}\n\n"

evidence_md_path = DOCS_ROOT / 'PF036_defense_evidence_matrix.md'
evidence_md_path.write_text(evidence_md, encoding='utf-8')

print('Defense evidence CSV:', evidence_path)
print('Defense evidence MD:', evidence_md_path)
evidence_df

## 7. README general de ejecución

In [ ]:
readme_text = '''# PFI MVP - Análisis asistido de RM lumbar mediante IA

## Alcance

MVP técnico demostrable para apoyo a la revisión de resonancia magnética lumbar. El sistema integra:

```text
datasets públicos
→ modelos de segmentación 2D axial y sagital
→ inferencia modular
→ agente de priorización
→ servicio Python FastAPI
→ backend Spring Boot
→ frontend React
→ revisión profesional
```

El sistema no realiza diagnóstico clínico autónomo.

## Componentes

### Servicio Python IA

Ruta:

```bash
ai_service/
```

Ejecución:

```bash
cd ai_service
uvicorn pfi_ai_service.api:app --host 0.0.0.0 --port 8000
```

Endpoints principales:

```text
GET  /health
GET  /models
POST /inference/sagittal
POST /inference/axial
POST /pipeline/run
GET  /agent/report/{run_id}
GET  /agent/regression-test
```

### Backend Spring Boot

Ruta:

```bash
backend_product_spring/
```

Ejecución:

```bash
cd backend_product_spring
mvn spring-boot:run
```

Endpoints principales:

```text
GET   /api/ai/health
GET   /api/ai/models
POST  /api/ai/pipeline/run
GET   /api/ai/agent/report/{runId}
PATCH /api/ai/review/{runId}
```

### Frontend React

Ruta:

```bash
frontend_product_ui/
```

Ejecución:

```bash
cd frontend_product_ui
npm install
npm run dev
```

`.env`:

```bash
VITE_API_BASE_URL=http://localhost:8080
```

## Política de uso

- Revisión profesional obligatoria.
- No diagnóstico autónomo.
- No reconstrucción 3D real en esta versión.
- Resultados interpretables como apoyo técnico y evidencia visual.

## Evidencia de desarrollo

Ver carpeta:

```bash
backlogProducto/
docs/
results/
```
'''

readme_path = REPO_ROOT / 'README_PRODUCTO_FINAL.md'
readme_path.write_text(readme_text, encoding='utf-8')
print('Wrote:', readme_path)

## 8. Checks finales y reporte

In [ ]:
checks = []

def add_check(name, ok, detail=''):
    checks.append({'check': name, 'ok': bool(ok), 'detail': str(detail)})

add_check('ai_service_exists', (REPO_ROOT / 'ai_service' / 'pfi_ai_service' / 'api.py').exists(), 'FastAPI api.py')
add_check('spring_backend_exists', (REPO_ROOT / 'backend_product_spring' / 'pom.xml').exists(), 'Spring pom.xml')
add_check('frontend_exists', (REPO_ROOT / 'frontend_product_ui' / 'package.json').exists(), 'React package.json')
add_check('artifact_inventory_written', artifact_path.exists(), artifact_path)
add_check('e2e_plan_written', e2e_path.exists(), e2e_path)
add_check('demo_script_written', demo_md_path.exists(), demo_md_path)
add_check('professional_validation_template_written', validation_md_path.exists(), validation_md_path)
add_check('documentation_update_plan_written', doc_update_md_path.exists(), doc_update_md_path)
add_check('defense_evidence_matrix_written', evidence_md_path.exists(), evidence_md_path)
add_check('readme_product_final_written', readme_path.exists(), readme_path)

checks_df = pd.DataFrame(checks)
checks_path = RESULTS_ROOT / 'PF031_PF038_checks.csv'
checks_df.to_csv(checks_path, index=False)
print('Checks:', checks_path)
print('All OK:', checks_df['ok'].all())
checks_df

In [ ]:
report = {
    'notebook': '34_PF031_PF038_e2e_demo_validation_finalization',
    'goal': 'prepare end-to-end demo, validation, documentation and final delivery evidence',
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'outputs': {
        'artifact_readiness_inventory_csv': str(artifact_path),
        'e2e_smoke_test_plan_csv': str(e2e_path),
        'e2e_smoke_tests_doc': str(e2e_md_path),
        'demo_script_csv': str(demo_path),
        'demo_script_doc': str(demo_md_path),
        'professional_validation_questions_csv': str(validation_path),
        'professional_validation_template_doc': str(validation_md_path),
        'documentation_update_plan_csv': str(doc_update_path),
        'documentation_update_plan_doc': str(doc_update_md_path),
        'defense_evidence_matrix_csv': str(evidence_path),
        'defense_evidence_matrix_doc': str(evidence_md_path),
        'readme_product_final': str(readme_path),
        'checks_csv': str(checks_path),
    },
    'summary': {
        'artifact_checks': int(len(artifact_df)),
        'artifact_checks_ok': bool(artifact_df['exists'].all()),
        'e2e_steps': int(len(e2e_df)),
        'demo_steps': int(len(demo_df)),
        'validation_questions': int(len(validation_df)),
        'documentation_updates': int(len(doc_update_df)),
        'defense_evidence_items': int(len(evidence_df)),
        'all_checks_ok': bool(checks_df['ok'].all()),
    },
    'methodological_policy': {
        'human_review_required': True,
        'not_clinical_diagnosis': True,
        'not_real_3d_reconstruction': True,
        'final_scope': 'MVP técnico demostrable con inferencia 2D multiplanar, agente, backend, frontend y revisión profesional'
    },
    'decision': 'PF031_PF038_final_delivery_checkpoint_ready'
}

report_path = RESULTS_ROOT / 'PF031_PF038_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')

closure_md = f'''# Cierre PF-031 a PF-038 - E2E, demo final, validación y cierre documental

Estado: cerrado.

## Resultado principal

El bloque cerró con la decisión:

```text
{report["decision"]}
```

## Salidas principales

- `docs/PF031_e2e_smoke_tests.md`
- `docs/PF032_final_demo_script.md`
- `docs/PF033_professional_validation_template.md`
- `docs/PF034_PF035_documentation_update_plan.md`
- `docs/PF036_defense_evidence_matrix.md`
- `README_PRODUCTO_FINAL.md`

## Checks

- artifact_checks_ok: {report["summary"]["artifact_checks_ok"]}
- all_checks_ok: {report["summary"]["all_checks_ok"]}

## Política metodológica

- human_review_required: true
- not_clinical_diagnosis: true
- not_real_3d_reconstruction: true

## Próximo paso

Con este bloque cerrado, corresponde realizar una pasada final manual:

1. Ejecutar demo end-to-end en entorno local.
2. Tomar capturas reales de frontend.
3. Hacer validación profesional usando la plantilla.
4. Actualizar capítulos 15 a 20 con evidencia final.
5. Preparar presentación/defensa.
'''

closure_path = BACKLOG_ROOT / 'PF031_PF038_resultados_cierre.md'
closure_path.write_text(closure_md, encoding='utf-8')

print('Report:', report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Closure:', closure_path)